# T-A2_Magnificent7 — 数据清洗与整理

| 项目   | 内容 |
|--------|------|
| 课程   | 数据分析与经济决策（ds2026） |
| 题目   | T-A2：美股"七姐妹"财务与股价全景对比 |
| 小组   | 第七组 |
| 成员   | 莫才有（7125210211）、刘帆（225210189）、王天正（325210247）、许玲敏（425210271）、高世杰（625210132）、曾垂健（725210291）、廖晟可（825210181） |
| GitHub | https://github.com/ZDC3/ds2026-G07-Magnificent-7 |
| Pages  | https://zdc3.github.io/ds2026-G07-Magnificent-7 |
| 日期   | 2026-05-16 |

## 任务说明
清洗股价数据，处理缺失值，计算收益率与风险指标，并保存清洗数据与指标结果，同时写入 SQLite 便于后续查询。

In [ ]:
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd

project_dir = Path('.').resolve()
data_raw_dir = project_dir / 'data_raw'
data_clean_dir = project_dir / 'data_clean'
data_clean_dir.mkdir(parents=True, exist_ok=True)

prices = pd.read_csv(data_raw_dir / 'prices_raw.csv', index_col=0, parse_dates=True)
financials = pd.read_csv(data_raw_dir / 'financials_raw.csv')

# 数据质量检查：缺失与重复
print('缺失值数量：')
print(prices.isna().sum())
print('重复日期数：', prices.index.duplicated().sum())

# 处理缺失值：前向填充，并删除仍然全缺失的行
prices = prices.sort_index().ffill()
prices = prices.dropna(how='all')

# 计算日度收益率
returns = prices.pct_change().dropna(how='all')

# 计算指标
def max_drawdown(series):
    cumulative = (1 + series).cumprod()
    rolling_max = cumulative.cummax()
    drawdown = (cumulative - rolling_max) / rolling_max
    return drawdown.min()

annual_return = returns.mean() * 252
annual_vol = returns.std() * np.sqrt(252)
rf_daily = 0.045 / 252
sharpe = (returns.mean() - rf_daily) / returns.std() * np.sqrt(252)
max_dd = returns.apply(max_drawdown)

metrics = pd.DataFrame({
    '年化收益率': annual_return,
    '年化波动率': annual_vol,
    '最大回撤': max_dd,
    '夏普比率': sharpe
})

# 相关系数矩阵（拓展）
correlation_matrix = returns.corr()

# 保存清洗结果
prices.to_csv(data_clean_dir / 'prices_clean.csv')
returns.to_csv(data_clean_dir / 'returns.csv')
metrics.to_csv(data_clean_dir / 'metrics.csv')
correlation_matrix.to_csv(data_clean_dir / 'correlation_matrix.csv')

# 写入 SQLite
db_path = data_clean_dir / 'm7.sqlite'
conn = sqlite3.connect(db_path)
prices.to_sql('prices_clean', conn, if_exists='replace')
returns.to_sql('returns', conn, if_exists='replace')
metrics.to_sql('metrics', conn, if_exists='replace')
financials.to_sql('financials_raw', conn, if_exists='replace', index=False)
correlation_matrix.to_sql('correlation_matrix', conn, if_exists='replace')
conn.close()

print('清洗完成，数据已保存到 data_clean/')
print('清洗后缺失值数量：')
print(prices.isna().sum())

## 结果解读
- 清洗后缺失是否显著下降？仍缺失的列说明原因。
- 日度收益率是否存在异常极值（可在后续分析中验证）。
- 指标计算是否合理（收益、波动、回撤、夏普方向一致）。